## Libraries and functions

In [2]:
# Operating system utilities (paths, environment variables, filesystem ops)
import os

# JSON handling for parsing and validating LLM responses
import json

# Path handling for filesystem-safe and cross-platform paths
from pathlib import Path

# Core data manipulation library used to structure LLM inputs
import pandas as pd

# OpenAI client used to call the LLM API
from openai import OpenAI

# SQLite driver used to read LLM inputs directly from the database
import sqlite3

# Estimates the number of tokens in a text using a conservative heuristic.
# This is intentionally rough but stable enough to avoid hitting hard limits.
def estimate_tokens_from_text(text: str) -> int:
    if not text:
        return 0
    return len(text) // 4


# Returns only a fraction of the input text.
# Used to progressively shrink large inputs before sending them to the LLM.
def trim_text_to_fraction(text: str, fraction: float) -> str:
    if not text or fraction >= 1.0:
        return text

    cut = int(len(text) * max(fraction, 0.0))
    return text[:cut]


# Trims the data block until the estimated prompt size fits within max_tokens.
# The idea is to keep as much data as possible without exceeding the limit.
def adaptive_trim_to_token_limit(system_prompt: str,user_prompt_template: str,data_block: str,
                                 max_tokens: int = 100_000,start_fraction: float = 0.99,step: float = 0.01):
    # Token cost of everything except the data payload.
    # This part does not change while trimming.
    base_prompt_tokens = (estimate_tokens_from_text(system_prompt)+ 
                          estimate_tokens_from_text(user_prompt_template.replace("{data_block}", "")))

    # Start by keeping almost all the data and gradually reduce it.
    fraction = start_fraction

    while fraction > 0:
        trimmed_block = trim_text_to_fraction(data_block, fraction)

        estimated_total_tokens = (base_prompt_tokens+ estimate_tokens_from_text(trimmed_block))

        # Verbose logging helps understand how aggressive the trimming is.
        print(f"Token check: fraction={fraction:.2f},estimated tokens={estimated_total_tokens}")

        # As soon as the estimate fits the limit, stop trimming.
        if estimated_total_tokens <= max_tokens:
            return trimmed_block, estimated_total_tokens, fraction

        fraction -= step

    # If even extreme trimming is not enough, fail explicitly.
    raise RuntimeError("Unable to reduce data_block under token limit.")


# Normalizes a keyword so it matches the naming convention
# used across SQL tables and exported artifacts.
def normalize_keyword(keyword: str) -> str:
    return keyword.lower().strip().replace(" ", "").replace("-", "")


# Converts a DataFrame into a readable text block for LLM consumption.
# Column names are included to preserve structural context.
def df_to_text(df: pd.DataFrame, name: str) -> str:
    if df is None or df.empty:
        return f"{name}: NOT PROVIDED OR EMPTY"

    return (f"{name} (all rows)\n"
            f"Columns: {list(df.columns)}\n"
            f"Data:\n"
            f"{df.to_string(index=False)}")


# Maps logical dataset roles to their corresponding CSV files.
# This is kept for compatibility with file-based pipelines.
def get_files(export_dir: Path, topic: str):
    return {"videos": ("video", "content", export_dir / f"{topic}_interesting_video.csv"),
            "reels":  ("reel",  "content", export_dir / f"{topic}_interesting_reel.csv"),
            "trends_top":    ("both", "query", export_dir / f"{topic}_google_top_ranked.csv"),
            "trends_rising": ("both", "query", export_dir / f"{topic}_google_rising_ranked.csv"),
            "publishing_day":  ("both", "temporal", export_dir / f"{topic}_publishing_day_trend_interesting.csv"),
            "publishing_hour": ("both", "temporal", export_dir / f"{topic}_publishing_hour_trend_interesting.csv"),}


# Loads all LLM-relevant inputs directly from the SQLite database.
# Each table is queried and packaged with metadata describing its role.
# Input : conn (sqlite3.Connection)  active SQLite connection
#         topic (str)               topic identifier used as table prefix
# Output: dict                      structured data for LLM consumption
#         int                       flag indicating availability of video content

def video_llm_inputs(conn: sqlite3.Connection, topic: str) -> dict:
    # Normalize topic into a compact prefix
    prefix = normalize_keyword(topic)

    # Define all required tables with their semantic role and SQL query
    tables = {"videos": {"role": "content",
                         "query": f"""SELECT title, description, source_score
                                      FROM {prefix}_interesting_video"""},
              "trends_top": {"role": "query",
                             "query": f"""SELECT query
                                          FROM {prefix}_google_top_ranked"""},
              "trends_rising": {"role": "query",
                                "query": f"""SELECT query
                                             FROM {prefix}_google_rising_ranked"""},
              "publishing_day": {"role": "temporal",
                                 "query": f"""SELECT *
                                              FROM {prefix}_interesting_publishing_day"""},
              "publishing_hour": {"role": "temporal",
                                  "query": f"""SELECT *
                                               FROM {prefix}_interesting_publishing_hour"""},
             }

    # Container for loaded data
    data = {}

    # Execute each query and store non-empty results
    for name, meta in tables.items():
        df = pd.read_sql_query(meta["query"], conn)
        if not df.empty:
            data[name] = {"role": meta["role"],"df": df}
            if name == "videos":
                print(f"Video LLM: loaded {len(df)} interesting videos")
            if name in {"trends_top", "trends_rising"}:
                print(f"Video LLM: loaded {len(df)} {name} queries")
            
    # Flag indicating whether video content is available
    has_videos=1
    if "videos" not in data:
        has_videos=0
        print("No video content available for LLM optimization.")
    
    # Return structured data and availability flag
    return data,has_videos


# Builds a textual data block for video-related LLM input.
# Each component is rendered according to its semantic role.
# Input : data (dict)   structured data loaded from the database
# Output: str           concatenated text block for LLM prompt

def video_data_block(data: dict) -> str:
    # Accumulator for text blocks
    blocks = []

    # Iterate through each loaded component
    for name, obj in data.items():
        role = obj["role"]

        if role == "content":
            df = obj["df"]
            blocks.append(df_to_text(df, "Proven_Long_Form"))

        elif role == "query":
            df = obj["df"]
            blocks.append(df_to_text(df, f"Search_{name.title()}"))

        elif role == "temporal":
            df = obj["df"]
            blocks.append(df_to_text(df, f"{name.title()}_Performance"))

        elif role == "analysis":
            blocks.append(f"\nEDA_report\n{obj['text']}\n")

    # Concatenate all blocks into a single string
    return "\n".join(blocks)


# Loads all reel-specific LLM inputs directly from the SQLite database.
# Mirrors the video loading logic but targets short-form content.
# Input : conn (sqlite3.Connection)  active SQLite connection
#         topic (str)               topic identifier used as table prefix
# Output: dict                      structured data for LLM consumption
#         int                       flag indicating availability of reel content

def reel_llm_inputs(conn: sqlite3.Connection, topic: str) -> dict:
    # Normalize topic into a compact prefix
    prefix = normalize_keyword(topic)

    # Define all required tables with their semantic role and SQL query
    tables = {"reels": {"role": "content",
                        "query": f"""SELECT title, description, source_score
                                     FROM {prefix}_interesting_reel"""},
              "trends_top": {"role": "query",
                             "query": f"""SELECT query
                                          FROM {prefix}_google_top_ranked"""},
              "trends_rising": {"role": "query",
                                "query": f"""SELECT query
                                             FROM {prefix}_google_rising_ranked"""},
              "publishing_day": {"role": "temporal",
                                 "query": f"""SELECT *
                                              FROM {prefix}_interesting_publishing_day"""},
              "publishing_hour": {"role": "temporal",
                                  "query": f"""SELECT *
                                               FROM {prefix}_interesting_publishing_hour"""},
             }

    # Container for loaded data
    data = {}

    # Execute each query and store non-empty results
    for name, meta in tables.items():
        df = pd.read_sql_query(meta["query"], conn)
        if not df.empty:
            data[name] = {"role": meta["role"],"df": df}
            if name == "reels":
                print(f"Reels LLM: loaded {len(df)} interesting reels")
            if name in {"trends_top", "trends_rising"}:
                print(f"Reels LLM: loaded {len(df)} {name} queries")

    # Flag indicating whether reel content is available
    has_reel=1
    if "reels" not in data:
        has_reel=0
        print("No reel content available for LLM optimization.")

    # Return structured data and availability flag
    return data,has_reel


# Builds a textual data block for reel-related LLM input.
# Each component is rendered according to its semantic role.
# Input : data (dict)   structured data loaded from the database
# Output: str           concatenated text block for LLM prompt

def reel_data_block(data: dict) -> str:
    # Accumulator for text blocks
    blocks = []

    # Iterate through each loaded component
    for name, obj in data.items():
        role = obj["role"]

        if role == "content":
            df = obj["df"]
            blocks.append(df_to_text(df, "Proven_Short_Form"))

        elif role == "query":
            df = obj["df"]
            blocks.append(df_to_text(df, f"Search_{name.title()}"))

        elif role == "temporal":
            df = obj["df"]
            blocks.append(df_to_text(df, f"{name.title()}_Performance"))

        elif role == "analysis":
            blocks.append(f"\nEDA_report\n{obj['text']}\n")

    # Concatenate all blocks into a single string
    return "\n\n".join(blocks)


# Renders the final LLM result into a human-readable text format.
# This function is used for logging, inspection and file export.
# Input : result (dict)  structured output produced by the LLM
# Output: str            formatted plain-text representation

def llm_result(result: dict) -> str:
    # Accumulator for output lines
    lines = []

    # Strategy decision
    decision = result.get("strategy_decision") or {}

    lines.append("LLM CONTENT STRATEGY DECISION")
    lines.append(f"Preferred content type : {decision.get('preferred_content_type', 'balanced')}")
    lines.append(f"Confidence level       : {decision.get('confidence_level', 'low')}")
    lines.append(f"Justification          : {decision.get('justification', 'No significant statistical relevance.')}")
    lines.append("")

    # Videos
    lines.append("VIDEO IDEAS")

    videos = result.get("videos", [])
    if not videos:
        lines.append("No video ideas generated.\n")
    else:
        for i, v in enumerate(videos, 1):
            lines.append(f"VIDEO {i}")
            lines.append(f"Title  : {v.get('title', '')}")
            lines.append(f"Tags   : {', '.join(v.get('tags', []))}")
            lines.append(f"Keywords : {', '.join(v.get('keywords', []))}")
            lines.append(f"Publish time : {v.get('best_publish_time', '')}")
            lines.append("")

    # Reels
    lines.append("REEL IDEAS")

    reels = result.get("reels", [])
    if not reels:
        lines.append("No reel ideas generated.\n")
    else:
        for i, r in enumerate(reels, 1):
            lines.append(f"REEL {i}")
            lines.append(f"Title  : {r.get('title', '')}")
            lines.append(f"Tags   : {', '.join(r.get('tags', []))}")
            lines.append(f"Keywords : {', '.join(r.get('keywords', []))}")
            lines.append(f"Publish time : {r.get('best_publish_time', '')}")
            lines.append("")

    # Return final text output
    return "\n".join(lines)


## Video results generation

In [1]:
# Executes the LLM optimization pipeline for long-form YouTube videos.
# This function prepares structured inputs, builds prompts, invokes the LLM,
# validates the response and returns structured video suggestions.
# Input : topic (str)                 topic identifier
#         storage_path (str | Path)   path to SQLite database
#         export_dir (str | Path)     directory for output artifacts
#         openai_key_file (str)       file containing the OpenAI API key
#         model (str)                 OpenAI model identifier
#         temperature (float)         sampling temperature for generation
#         enable_llm (bool)           flag to enable or disable LLM execution
# Output: dict                        structured LLM output and metadata

def llm_video(topic: str,storage_path: str | Path,export_dir: str | Path,openai_key_file: str,
              model: str = "gpt-4.1-mini",temperature: float = 0.4,enable_llm: bool = True,):
    
    # Normalize topic for consistent naming
    topic_norm = normalize_keyword(topic)

    # Resolve paths to absolute locations
    export_dir = Path(export_dir).resolve()
    DB_PATH = Path(storage_path).resolve()

    # Validate required paths
    if not export_dir.exists():
        raise FileNotFoundError(export_dir)
    if not DB_PATH.exists():
        raise FileNotFoundError(DB_PATH)

    # Load OpenAI API key from file
    key_path = Path.cwd() / openai_key_file
    openai_key = key_path.read_text().strip()

    # Set API key in environment
    os.environ["OPENAI_API_KEY"] = openai_key

    # Initialize OpenAI client
    client = OpenAI()

    # Load all LLM-relevant video inputs from database
    with sqlite3.connect(DB_PATH) as conn:
        data, has_video = video_llm_inputs(conn, topic)

    # Abort if no video data is available
    if has_video==0:
        print("No videos collected")
        return
        
    # Build textual data block for LLM context
    data_block = video_data_block(data)

    # Define system-level instructions for the LLM
    system_prompt = ("You are an Elite YouTube Algorithm Strategist & Data Scientist. "
                     "Your capability is to synthesize multi-source analytics into high-ROI content specifications. "
                     "You output ONLY valid, raw JSON without markdown formatting.")

    # Define user prompt template for video generation
    user_prompt_video = """
    ### MISSION
    Analyze the datasets provided below in the <DATA_CONTEXT> tag.
    Synthesize these signals to generate exactly 3 Long-form Video concepts.
    
    ### DATASETS EXPLANATION
    1. [Proven_Long_Form]: Core topics that consistently guarantee long-term retention.
    2. [Search_Intent_Stable]: High-confidence keywords for discoverability and evergreen traffic.
    3. [Search_Intent_Rising]: Secondary angles that can increase click-through rate while preserving topic stability.
    4. [Day_Performance]: Statistical distribution of views by day.
    5. [Hour_Performance]: Statistical distribution of views by hour.
    
    ### PRIORITY SIGNAL (VERY IMPORTANT)
    Some content entries include a field called "source_score":
    - source_score = 2 → topic appears in BOTH Best Content and Top Opportunity datasets (highest priority)
    - source_score = 1 → topic appears in only one dataset
    
    You MUST prioritize topics with source_score = 2 when selecting the core foundation for video concepts.
    Use source_score = 1 only if insufficient high-priority topics are available.
    
    ### SYNTHESIS LOGIC (Step-by-Step)
    1. Foundation:
       Select a core topic from Proven_Long_Form, prioritizing entries with source_score = 2.
    2. SEO Optimization:
       Structure the video concept around Search_Intent_Stable keywords.
       Keywords MUST be realistic candidates for Google search results related to the provided queries.
    3. CTR Enhancement:
       Carefully integrate Search_Intent_Rising elements as sub-angles, not as the main topic.
    4. Timing:
       - If best day/hour is less than 15% above average → "There is no preferred time or day"
       - Else → "Day at HH:00", for each video different a timing
    
    ### SEARCH CONSISTENCY CONSTRAINT
    All titles, tags, and keywords MUST align with the semantic intent of the Google queries present in the input data.
    The generated concepts must plausibly rank or appear in search results for those queries.
    
    ### CONSTRAINTS
    - NO Hallucinations
    - NO short-form concepts
    - NO speculative topics
    - Strict JSON only
    - All information given must be in english
    
    ### OUTPUT JSON SCHEMA
    {{
      "videos": [
        {{ "title": "...", "tags": [], "keywords": [], "best_publish_time": "..." }}
      ]
    }}
    
    ### <DATA_CONTEXT>
    {data_block}
    ### </DATA_CONTEXT>
    """

    # Adaptively trim data block to respect token limits
    data_block, estimated_tokens, used_fraction = adaptive_trim_to_token_limit(
                                                                               system_prompt=system_prompt,
                                                                               user_prompt_template=user_prompt_video,
                                                                               data_block=data_block,
                                                                               max_tokens=100_000,
                                                                               start_fraction=1.00,
                                                                               step=0.01)

    # Finalize user prompt with trimmed data
    final_user_prompt = user_prompt_video.format(data_block=data_block)

    # Return metadata only if LLM execution is disabled
    if not enable_llm:
        return {"topic": topic_norm,"estimated_input_tokens": estimated_tokens,"data_fraction_used": used_fraction,
                "output_file": None}

    # Invoke OpenAI Chat Completion API
    response = client.chat.completions.create(model=model,temperature=temperature,
                                              messages=[{"role": "system", "content": system_prompt},
                                                        {"role": "user", "content": final_user_prompt}])

    # Extract raw LLM output
    raw = response.choices[0].message.content.strip()

    # Parse JSON output
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        raise ValueError(f"Invalid JSON returned:\n{raw}")

    # Extract generated video concepts
    videos = parsed.get("videos", [])

    # Build human-readable summary lines
    lines = [f"SUGGESTIONS FOR {topic.title()} — VIDEOS\n"]

    for i, v in enumerate(videos, 1):
        lines.extend([f"VIDEO {i}",
                      f"Title: {v.get('title', '')}",
                      f"Tags: {', '.join(v.get('tags', []))}",
                      f"Keywords to put in description: {', '.join(v.get('keywords', []))}",
                      f"Best publish time: {v.get('best_publish_time', '')}\n"])

    # Return structured result and metadata
    return {
        "type": "video",
        "videos": videos,
        "estimated_input_tokens": estimated_tokens,
        "data_fraction_used": used_fraction
    }

NameError: name 'Path' is not defined

## Reels results generation

In [1]:
# Executes the LLM optimization pipeline for short-form YouTube reels.
# This function prepares structured inputs, builds prompts, invokes the LLM,
# validates the response and returns structured reel suggestions.
# Input : topic (str)                 topic identifier
#         storage_path (str | Path)   path to SQLite database
#         export_dir (str | Path)     directory for output artifacts
#         openai_key_file (str)       file containing the OpenAI API key
#         model (str)                 OpenAI model identifier
#         temperature (float)         sampling temperature for generation
#         enable_llm (bool)           flag to enable or disable LLM execution
# Output: dict                        structured LLM output and metadata

def llm_reel(topic: str,storage_path: str | Path,export_dir: str | Path,openai_key_file: str,
             model: str = "gpt-4.1-mini",temperature: float = 0.4,enable_llm: bool = True,):
    # Normalize topic for consistent naming
    topic_norm = normalize_keyword(topic)

    # Resolve paths to absolute locations
    export_dir = Path(export_dir).resolve()
    DB_PATH = Path(storage_path).resolve()

    # Validate required paths
    if not export_dir.exists():
        raise FileNotFoundError(export_dir)
    if not DB_PATH.exists():
        raise FileNotFoundError(DB_PATH)

    # Load OpenAI API key from file
    key_path = Path.cwd() / openai_key_file
    openai_key = key_path.read_text().strip()

    # Set API key in environment
    os.environ["OPENAI_API_KEY"] = openai_key

    # Initialize OpenAI client
    client = OpenAI()

    # Load all LLM-relevant reel inputs from database
    with sqlite3.connect(DB_PATH) as conn:
        data, has_reel = reel_llm_inputs(conn, topic)

    # Abort if no reel data is available
    if has_reel==0:
        print("No reels collected")
        return

    # Build textual data block for LLM context
    data_block = reel_data_block(data)

    # Define system-level instructions for the LLM
    system_prompt = ("You are an Elite YouTube Algorithm Strategist & Data Scientist. "
                     "Your capability is to synthesize multi-source analytics into high-ROI content specifications. "
                     "You output ONLY valid, raw JSON without markdown formatting.")

    # Define user prompt template for reel generation
    user_prompt_reel = """
    ### MISSION
    Analyze the datasets provided below in the <DATA_CONTEXT> tag.
    Synthesize these signals to generate exactly 3 Short-form Reel concepts.
    
    ### DATASETS EXPLANATION
    1. [Proven_Short_Form]: Hooks, visual styles, and narrative patterns that empirically drive short-form engagement.
    2. [Search_Intent_Stable]: Search queries that ensure semantic alignment with established user demand.
    3. [Search_Intent_Rising]: Emerging queries indicating novelty, urgency, or sudden interest spikes.
    4. [Day_Performance]: Statistical distribution of views by day.
    5. [Hour_Performance]: Statistical distribution of views by hour.
    
    ### PRIORITY SIGNAL (VERY IMPORTANT)
    Some entries in Proven_Short_Form include a field called "source_score":
    - source_score = 2 → hook appears in BOTH Best Content and Top Opportunity datasets (highest priority)
    - source_score = 1 → hook appears in only one dataset
    
    You MUST prioritize hooks with source_score = 2 when selecting the foundation for reel concepts.
    Use source_score = 1 only if insufficient high-priority hooks are available.
    
    ### SYNTHESIS LOGIC (Step-by-Step)
    1. Foundation:
       Select a proven hook or visual pattern from Proven_Short_Form, prioritizing source_score = 2.
    2. SEO Alignment:
       Reframe the concept using Search_Intent_Stable keywords.
       Selected keywords MUST plausibly appear in Google search results for the given queries.
    3. Novelty Injection:
       Enhance the hook using Search_Intent_Rising angles without altering semantic relevance.
    4. Timing:
       - If best day/hour is less than 15% above average → "There is no preferred time or day"
       - Else → "Day at HH:00", for each reel choose different timing
    
    ### SEARCH CONSISTENCY CONSTRAINT
    All titles, tags, and keywords MUST be semantically consistent with at least one Google search query contained in the provided Search_Intent datasets.
    Do NOT introduce terms that would not reasonably match a Google search result for those queries.
    
    ### CONSTRAINTS
    - NO Hallucinations
    - NO assumptions beyond provided data
    - Strict JSON only
    - Do NOT output video concepts
    - All information given must be in english
    
    ### OUTPUT JSON SCHEMA
    {{
      "reels": [
        {{ "title": "...", "tags": [], "keywords": [], "best_publish_time": "..." }}
      ]
    }}
    
    ### <DATA_CONTEXT>
    {data_block}
    ### </DATA_CONTEXT>
    """

    # Adaptively trim data block to respect token limits
    data_block, estimated_tokens, used_fraction = adaptive_trim_to_token_limit(
                                                                               system_prompt=system_prompt,
                                                                               user_prompt_template=user_prompt_reel,
                                                                               data_block=data_block,
                                                                               max_tokens=100_000,
                                                                               start_fraction=1.00,
                                                                               step=0.01)

    # Finalize user prompt with trimmed data
    final_user_prompt = user_prompt_reel.format(data_block=data_block)

    # Return metadata only if LLM execution is disabled
    if not enable_llm:
        return {"topic": topic_norm,
                "estimated_input_tokens": estimated_tokens,
                "data_fraction_used": used_fraction,
                "output_file": None}

    # Invoke OpenAI Chat Completion API
    response = client.chat.completions.create(model=model,temperature=temperature,
                                              messages=[{"role": "system", "content": system_prompt},
                                                        {"role": "user", "content": final_user_prompt}])

    # Extract raw LLM output
    raw = response.choices[0].message.content.strip()

    # Parse JSON output
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        raise ValueError(f"Invalid JSON returned:\n{raw}")

    # Extract generated reel concepts
    reels = parsed.get("reels", [])

    # Build human-readable summary lines
    lines = [f"SUGGESTIONS FOR {topic.title()} — REELS\n"]

    for i, r in enumerate(reels, 1):
        lines.extend([f"REEL {i}",
                      f"Title: {r.get('title', '')}",
                      f"Tags: {', '.join(r.get('tags', []))}",
                      f"Keywords to put in description: {', '.join(r.get('keywords', []))}",
                      f"Best publish time: {r.get('best_publish_time', '')}\n"])

    # Return structured result and metadata
    return {"type": "reel",
            "reels": reels,
            "estimated_input_tokens": estimated_tokens,
            "data_fraction_used": used_fraction}


NameError: name 'Path' is not defined

## Content strategy generation

In [ ]:
# Executes an LLM-based content strategy decision grounded in statistical evidence.
# This function evaluates precomputed significance tests to determine whether
# video content, reel content, or a balanced strategy should be prioritized.
# Input : topic (str)                 topic identifier
#         storage_path (str | Path)   path to SQLite database
#         openai_key_file (str)       file containing the OpenAI API key
#         model (str)                 OpenAI model identifier
#         temperature (float)         sampling temperature for generation
#         alpha (float)               statistical significance threshold
#         enable_llm (bool)           flag to enable or disable LLM execution
# Output: dict                        structured strategy decision and metadata

def llm_content_strategy_decision(topic: str,storage_path: str | Path,openai_key_file: str,
                                  model: str = "gpt-4.1-mini",temperature: float = 0.2,
                                  alpha: float = 0.05,enable_llm: bool = True):

    # Normalize topic for consistent naming
    topic_norm = normalize_keyword(topic)

    # Resolve database path
    DB_PATH = Path(storage_path).resolve()

    # Validate database existence
    if not DB_PATH.exists():
        raise FileNotFoundError(DB_PATH)

    # Load OpenAI API key from file
    key_path = Path.cwd() / openai_key_file
    os.environ["OPENAI_API_KEY"] = key_path.read_text().strip()

    # Initialize OpenAI client
    client = OpenAI()

    # Read precomputed significance test results from SQLite
    with sqlite3.connect(DB_PATH) as conn:
        table = f"{topic_norm}_significance_tests"
        try:
            df = pd.read_sql_query(f"SELECT metric, p_value, diff_pct FROM {table}",conn)
        except Exception:
            df = pd.DataFrame()


    # If no statistical results are available, return a balanced low-confidence decision
    if df.empty:
        return {
            "type": "strategy_decision",
            "strategy_decision": {
                "preferred_content_type": "balanced",
                "confidence_level": "low",
                "justification": "No significant statistical relevance.",
                "evidence_summary": {
                    "significant_metrics_count": 0,
                    "metrics_favoring_videos": 0,
                    "metrics_favoring_reels": 0
                }
            }
        }

    # Prepare a structured textual summary of statistical tests
    lines = ["STATISTICAL_TESTS (Welch T-Test: Reels vs Videos)",
             "",
             f"Significance threshold: alpha = {alpha}",
             "",
             "Results:"]

    for _, r in df.iterrows():
        lines.append(f"- {r['metric']}: "
                     f"p_value={r['p_value']:.4f}, "
                     f"diff_pct={r['diff_pct']:+.1f}%")

    # Combine all lines into a single evidence block
    stats_block = "\n".join(lines)

    # Define system-level instructions for statistical reasoning
    system_prompt = ("You are a statistical decision analyst. "
                     "You MUST output ONLY raw JSON. "
                     "Do NOT use markdown or code fences.")

    # Define user prompt with explicit decision constraints
    user_prompt = f"""
    ### MISSION
    Based on the statistical evidence provided below, decide whether it is
    statistically justified to prioritize VIDEO content, REEL content,
    or adopt a BALANCED strategy.
    
    ### DECISION CONSTRAINTS
    - Use ONLY the provided statistical tests
    - Do NOT recompute statistics
    - Do NOT assume missing data
    - p_value < {alpha} indicates statistical significance
    - diff_pct > 0 favors REELS
    - diff_pct < 0 favors VIDEOS
    
    ### DECISION LOGIC
    - If most significant metrics favor one content type → prefer it
    - If evidence is mixed or weak (non significant) → balanced strategy
    - Justify your decision clearly
    
    ### OUTPUT JSON SCHEMA
    {{
      "preferred_content_type": "videos | reels | balanced",
      "confidence_level": "high | medium | low",
      "justification": "...",
      "evidence_summary": {{
        "significant_metrics_count": int,
        "metrics_favoring_videos": int,
        "metrics_favoring_reels": int
      }}
    }}
    
    ### STATISTICAL_EVIDENCE
    {stats_block}
    """

    # Return metadata only if LLM execution is disabled
    if not enable_llm:
        return {
            "type": "strategy_decision",
            "strategy_decision": None
        }

    # Invoke OpenAI Chat Completion API
    response = client.chat.completions.create(model=model,temperature=temperature,
                                              messages=[{"role": "system", "content": system_prompt},
                                                        {"role": "user", "content": user_prompt}])

    # Extract raw LLM output
    raw = response.choices[0].message.content.strip()

    # Remove accidental markdown wrappers if present
    raw_clean = raw
    if raw_clean.startswith("```"):
        raw_clean = raw_clean.strip("`").replace("json", "", 1).strip()

    try:
        parsed = json.loads(raw_clean)
    except Exception:
        parsed = None

    # Enforce a safe balanced decision if parsing fails
    if not isinstance(parsed, dict) or "preferred_content_type" not in parsed:
        parsed = {
            "preferred_content_type": "balanced",
            "confidence_level": "low",
            "justification": "No significant statistical relevance.",
            "evidence_summary": {
                "significant_metrics_count": 0,
                "metrics_favoring_videos": 0,
                "metrics_favoring_reels": 0
            }
        }

    # Return final strategy decision
    return {
        "type": "strategy_decision",
        "strategy_decision": parsed
    }


## LLM pipeline

In [2]:
from pathlib import Path

# Orchestrates the complete LLM optimization pipeline.
# This function coordinates strategy decision, video generation and reel generation,
# aggregates results, renders a human-readable report and writes it to disk.
# Input : topic (str)                 topic identifier
#         storage_path (str | Path)   path to SQLite database
#         export_dir (str | Path)     directory for output artifacts
#         openai_key_file (str)       file containing the OpenAI API key
#         model (str)                 OpenAI model identifier
#         temperature (float)         sampling temperature for generation
#         enable_llm (bool)           flag to enable or disable LLM execution
# Output: dict                        aggregated LLM results and metadata

def run_llm_optimization(topic: str,storage_path: str | Path,export_dir: str | Path, openai_key_file: str,
                         model: str = "gpt-4.1-mini", temperature: float = 0.4, enable_llm: bool = True):

    # Normalize export directory path
    export_dir = Path(export_dir)

    # Normalize topic for consistent naming
    topic_norm = normalize_keyword(topic)

    # Initialize final result container
    final_result = {"topic": topic_norm,
                    "strategy_decision": None,
                    "videos": [],
                    "reels": [],
                    "metadata": {}}


    # Strategy decision phase

    try:
        decision = llm_content_strategy_decision(topic=topic,storage_path=storage_path,
                                                     openai_key_file=openai_key_file,model=model,
                                                     enable_llm=enable_llm)
        final_result["strategy_decision"] = decision["strategy_decision"]
    except Exception as e:
        final_result["strategy_decision"] = {"preferred_content_type": "balanced",
                                             "confidence_level": "low",
                                             "justification": f"Decision failed: {e}"}


    # Video generation phase

    try:
        video_result = llm_video(topic=topic,storage_path=storage_path,export_dir=export_dir,
                                     openai_key_file=openai_key_file,model=model,
                                     temperature=temperature,enable_llm=enable_llm)
        if video_result:
            final_result["videos"] = video_result.get("videos", [])
            final_result["metadata"]["video_tokens"] = video_result.get("estimated_input_tokens")
    except Exception as e:
        print(f"Video generation failed: {e}")


    # Reel generation phase
    try:
        reel_result = llm_reel(topic=topic,storage_path=storage_path,export_dir=export_dir,
                                   openai_key_file=openai_key_file,model=model,temperature=temperature,
                                   enable_llm=enable_llm)
        if reel_result:
            final_result["reels"] = reel_result.get("reels", [])
            final_result["metadata"]["reel_tokens"] = reel_result.get("estimated_input_tokens")
    except Exception as e:
        print(f"Reel generation failed: {e}")

    # Output rendering
    
    # Build output file path
    output_file = export_dir / f"{topic_norm}_llm_result.txt"

    # Render final result as plain text
    output_text = llm_result(final_result)

    # Write output to disk
    output_file.write_text(output_text, encoding="utf-8")

    # Return aggregated result
    return final_result

NameError: name 'Path' is not defined